# Calories Guard — DeepSeek fine-tune

Fine-tune `DeepSeek-R1-Distill-Qwen-1.5B` on a synthetic Thai nutrition Q&A dataset scoped to the Calories Guard app (Thai food calories, macronutrients, diet advice, BMR math). The adapter is then plugged into the backend via the `LOCAL_MODEL_PATH` / `LOCAL_ADAPTER_PATH` env vars and selected with `LLM_PROVIDER=local`.

Pipeline:
1. **Dataset** — ~60 curated Thai Q&A pairs about the app's domain.
2. **Thai word segmentation with an MLP** — since Thai has no spaces, we train a small MLP on character-window features to predict word boundaries. Used as a sanity step before feeding data into the LLM tokenizer. (Educational; the tokenizer ultimately handles subwords itself.)
3. **LoRA fine-tune** — low-rank adapters on the base model so it runs on Colab free tier.
4. **Q↔A alignment visualization** — per epoch, we generate answers on a held-out split and plot the token-overlap percentage between the model's answer and the reference answer, alongside training loss. Straight line up = model is learning the domain; flat line = underfitting.

Run top-to-bottom on a GPU runtime.

## Setup

In [ ]:
!pip install -q transformers==4.44.2 peft==0.13.0 accelerate==0.34.2 bitsandbytes==0.44.0 datasets==3.0.0 pythainlp==5.0.4 matplotlib scikit-learn

In [ ]:
import os
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. App-scoped Thai nutrition Q&A dataset

Hand-curated so every example reflects questions a Calories Guard user would actually ask — Thai food calories, BMR/TDEE, portion sizes, allergy flags, the app's own features. Keep this file version-controlled; re-run fine-tune when it grows.

In [ ]:
QA_PAIRS = [
    {'q': 'ข้าวผัดกุ้งจานหนึ่งกี่แคลอรี่', 'a': 'ข้าวผัดกุ้ง 1 จาน (ประมาณ 350 กรัม) ให้พลังงานราว 550-650 กิโลแคลอรี่ ขึ้นกับน้ำมันและปริมาณกุ้ง.'},
    {'q': 'ต้มยำกุ้งน้ำใสกี่แคลอรี่', 'a': 'ต้มยำกุ้งน้ำใส 1 ถ้วย (250 กรัม) ประมาณ 90-120 กิโลแคลอรี่ ถ้าเป็นน้ำข้นจะเพิ่มเป็น 250-300.'},
    {'q': 'ผัดไทยกุ้งสดกี่แคล', 'a': 'ผัดไทยกุ้งสด 1 จาน (350 กรัม) ให้พลังงาน 500-600 กิโลแคลอรี่ เพิ่มถั่วลิสงและน้ำมันจะขึ้นอีก.'},
    {'q': 'ส้มตำไทยไม่ใส่ถั่วกี่แคล', 'a': 'ส้มตำไทย 1 จาน (250 กรัม) ไม่ใส่ถั่วลิสง ประมาณ 130-170 กิโลแคลอรี่.'},
    {'q': 'ข้าวเหนียวหมูปิ้ง 2 ไม้ แคลเท่าไหร่', 'a': 'ข้าวเหนียว 1 ห่อ (100 กรัม) ~240 กิโลแคลอรี่ หมูปิ้ง 2 ไม้ ~160 กิโลแคลอรี่ รวม ~400 กิโลแคลอรี่.'},
    {'q': 'ก๋วยเตี๋ยวต้มยำหมูสับกี่แคล', 'a': 'ก๋วยเตี๋ยวต้มยำหมูสับ 1 ชาม (400 กรัม) ประมาณ 380-450 กิโลแคลอรี่.'},
    {'q': 'กะเพราหมูสับไข่ดาวกี่แคลอรี่', 'a': 'กะเพราหมูสับราดข้าว 1 จาน + ไข่ดาว 1 ฟอง ประมาณ 600-700 กิโลแคลอรี่.'},
    {'q': 'แกงเขียวหวานไก่กับข้าวกี่แคล', 'a': 'แกงเขียวหวานไก่ 1 ถ้วย (200 กรัม) + ข้าวสวย 1 ถ้วย (150 กรัม) ประมาณ 500-600 กิโลแคลอรี่.'},
    {'q': 'โจ๊กหมูใส่ไข่กี่แคล', 'a': 'โจ๊กหมูใส่ไข่ 1 ชาม (400 กรัม) ประมาณ 280-350 กิโลแคลอรี่.'},
    {'q': 'ข้าวมันไก่จานหนึ่งมีกี่แคลอรี่', 'a': 'ข้าวมันไก่ 1 จาน (350 กรัม) ประมาณ 550-650 กิโลแคลอรี่ หนังติดมันจะเพิ่มอีก.'},
    {'q': 'ลาบหมูกี่แคล', 'a': 'ลาบหมู 1 จาน (150 กรัม) ประมาณ 200-260 กิโลแคลอรี่ ถ้าใส่ข้าวคั่วเพิ่มอีก 30-50.'},
    {'q': 'น้ำตกเนื้อกี่แคล', 'a': 'น้ำตกเนื้อ 1 จาน (150 กรัม) ประมาณ 230-280 กิโลแคลอรี่.'},
    {'q': 'ไข่เจียวหมูสับกี่แคลอรี่', 'a': 'ไข่เจียวหมูสับใช้ไข่ 2 ฟอง + หมูสับ 50 กรัม ทอดในน้ำมัน ประมาณ 350-420 กิโลแคลอรี่.'},
    {'q': 'โรตีชีสกี่แคล', 'a': 'โรตีชีส 1 แผ่น (150 กรัม) ประมาณ 380-450 กิโลแคลอรี่ ขึ้นกับปริมาณน้ำตาลและนม.'},
    {'q': 'ชานมไข่มุกแก้วเดียวกี่แคล', 'a': 'ชานมไข่มุก 1 แก้ว (500 มล.) ประมาณ 300-500 กิโลแคลอรี่ ไข่มุกและน้ำตาลคือตัวแปรหลัก.'},
    {'q': 'กาแฟเย็นหวานน้อยกี่แคล', 'a': 'กาแฟเย็นหวานน้อย 1 แก้ว (400 มล.) ประมาณ 150-200 กิโลแคลอรี่.'},
    {'q': 'น้ำมะพร้าวแก้วเดียวกี่แคล', 'a': 'น้ำมะพร้าวสด 1 ลูก (300 มล.) ประมาณ 60-80 กิโลแคลอรี่ ไม่รวมเนื้อ.'},
    {'q': 'ข้าวไข่เจียวกี่แคล', 'a': 'ข้าวไข่เจียว (ไข่ 2 ฟอง + ข้าว 1 จาน) ประมาณ 450-520 กิโลแคลอรี่.'},
    {'q': 'สลัดอกไก่ย่างกี่แคล', 'a': 'สลัดผักอกไก่ย่าง 1 จาน (300 กรัม) น้ำสลัดใส ประมาณ 220-290 กิโลแคลอรี่.'},
    {'q': 'ข้าวกล่องอกไก่ต้มมีโปรตีนกี่กรัม', 'a': 'อกไก่ต้ม 150 กรัม มีโปรตีนประมาณ 35-40 กรัม ไขมันต่ำประมาณ 3 กรัม.'},
    {'q': 'ไข่ต้ม 1 ฟองมีโปรตีนกี่กรัม', 'a': 'ไข่ไก่ต้ม 1 ฟอง (50 กรัม) มีโปรตีน 6-7 กรัม ไขมัน 5 กรัม ~70 กิโลแคลอรี่.'},
    {'q': 'ข้าวสวย 1 ทัพพีกี่แคล', 'a': 'ข้าวสวย 1 ทัพพี (~60 กรัม) ประมาณ 75-85 กิโลแคลอรี่.'},
    {'q': 'ข้าวกล้อง 1 ถ้วยกี่แคล', 'a': 'ข้าวกล้อง 1 ถ้วย (150 กรัม) ประมาณ 170-200 กิโลแคลอรี่ มีใยอาหารสูงกว่าข้าวขาว.'},
    {'q': 'แอปเปิ้ลลูกกลางมีกี่แคล', 'a': 'แอปเปิ้ลลูกกลาง (~180 กรัม) ประมาณ 90-100 กิโลแคลอรี่ มีใยอาหาร 4 กรัม.'},
    {'q': 'กล้วยหอม 1 ลูกกี่แคล', 'a': 'กล้วยหอม 1 ลูก (~120 กรัม) ประมาณ 105-115 กิโลแคลอรี่ มีโพแทสเซียมสูง.'},
    {'q': 'นมวัวสด 250 มล. กี่แคล', 'a': 'นมวัวสดไขมันเต็ม 250 มล. ประมาณ 150-160 กิโลแคลอรี่ ถ้าไขมันต่ำ ~100.'},
    {'q': 'BMR คืออะไร คำนวณยังไง', 'a': 'BMR คือพลังงานที่ร่างกายใช้ขณะพัก สูตร Mifflin-St Jeor ผู้ชาย: 10×น้ำหนัก + 6.25×ส่วนสูง - 5×อายุ + 5, ผู้หญิงลบ 161 แทนบวก 5.'},
    {'q': 'TDEE ต่างจาก BMR ยังไง', 'a': 'TDEE = BMR × ค่ากิจกรรม (1.2 ไม่ออกกำลัง, 1.375 เบา, 1.55 ปานกลาง, 1.725 หนัก). เป็นแคลอรี่ที่ร่างกายต้องใช้จริงต่อวัน.'},
    {'q': 'อยากลดน้ำหนัก 1 กิโลต่อสัปดาห์ ต้องลดแคลวันละเท่าไหร่', 'a': 'ไขมัน 1 กิโลกรัม ≈ 7,700 กิโลแคลอรี่ ต้องขาดดุลพลังงานประมาณ 1,100 กิโลแคลอรี่ต่อวันในการลด 1 กก./สัปดาห์ ซึ่งเร็วเกินไปและยากยั่งยืน แนะนำ 0.5 กก./สัปดาห์ (ขาดดุล 550 kcal/วัน).'},
    {'q': 'กินคาร์บน้อยคืนละเท่าไหร่ถึงจะดี', 'a': 'สำหรับคีโต คาร์บสุทธิ < 50 กรัม/วัน, โลว์คาร์บทั่วไป 50-130 กรัม/วัน, ไม่แนะนำต่ำกว่า 20 กรัม/วันโดยไม่ปรึกษาแพทย์.'},
    {'q': 'โปรตีนวันละเท่าไหร่สำหรับคนออกกำลังกาย', 'a': 'คนทั่วไป 0.8 ก./กก. ออกกำลังกายคาร์ดิโอ 1.2-1.4 ก./กก. เวทเทรนนิ่งเพิ่มกล้าม 1.6-2.2 ก./กก.'},
    {'q': 'ไขมันอิ่มตัวกับไขมันไม่อิ่มตัวต่างกันยังไง', 'a': 'ไขมันอิ่มตัว (เนย, มันสัตว์) เป็นของแข็งที่อุณหภูมิห้องและเพิ่ม LDL. ไขมันไม่อิ่มตัว (น้ำมันมะกอก, ถั่ว, ปลา) ช่วยลด LDL ควรจำกัดไขมันอิ่มตัว < 10% ของแคลอรี่ทั้งวัน.'},
    {'q': 'กินไฟเบอร์วันละเท่าไหร่', 'a': 'ผู้ใหญ่ควรได้ไฟเบอร์ 25-38 กรัม/วัน ส่วนใหญ่จากผัก ผลไม้ ธัญพืชไม่ขัดสี ช่วยขับถ่ายและควบคุมน้ำตาล.'},
    {'q': 'ดื่มน้ำเปล่าวันละกี่ลิตรดี', 'a': 'ผู้ใหญ่ 2-2.5 ลิตรต่อวัน หรือ 30-35 มล./กก.น้ำหนักตัว เพิ่มขึ้นเมื่ออากาศร้อนหรือออกกำลังกาย.'},
    {'q': 'วิ่งเผาผลาญกี่แคลต่อชั่วโมง', 'a': 'วิ่ง 8 กม./ชม. (6:45 min/km) เผาผลาญ 500-650 กิโลแคลอรี่/ชั่วโมง ขึ้นกับน้ำหนักตัว. 60 กก. ≈ 540 kcal.'},
    {'q': 'เดินเร็ว 30 นาทีเผาผลาญเท่าไหร่', 'a': 'เดินเร็ว 6 กม./ชม. เผาผลาญ ~4 MET × 60 กก. × 0.5 ชม. ≈ 120 กิโลแคลอรี่.'},
    {'q': 'ว่ายน้ำ 45 นาทีเผาผลาญเท่าไหร่', 'a': 'ว่ายน้ำฟรีสไตล์ระดับกลาง ~8 MET × 60 กก. × 0.75 ชม. ≈ 360 กิโลแคลอรี่.'},
    {'q': 'ยกน้ำหนัก 1 ชั่วโมงเผาผลาญกี่แคล', 'a': 'เวทเทรนนิ่งหนักปานกลาง ~6 MET × 60 กก. × 1 ชม. ≈ 360 กิโลแคลอรี่ และยังเพิ่ม EPOC หลังออกกำลัง.'},
    {'q': 'กินอิ่มแล้วยังหิว ทำยังไง', 'a': 'เพิ่มโปรตีนและไฟเบอร์ในมื้อ, ดื่มน้ำก่อนกิน, นอนให้พอ, ตรวจว่ากินแคลอรี่รวมต่ำเกินไปไหม (ต่ำกว่า BMR ทำให้หิวตลอด).'},
    {'q': 'อาหารว่างกลางคืนแคลต่ำแนะนำอะไร', 'a': 'โยเกิร์ตกรีกไม่หวาน 100 กรัม (~60 kcal), แอปเปิ้ลครึ่งลูก (~50 kcal), ไข่ต้ม 1 ฟอง (~70 kcal), ถั่วเอดามาเม่ 50 กรัม (~60 kcal).'},
    {'q': 'น้ำตาลวันละเท่าไหร่ไม่เกินดี', 'a': 'WHO แนะนำน้ำตาลเติมไม่เกิน 10% ของแคลอรี่รวม เหมาะสมยิ่งกว่าคือ < 5% (ผู้หญิง ~25 กรัม/วัน, ผู้ชาย ~36 กรัม/วัน).'},
    {'q': 'ผลไม้ชนิดไหนน้ำตาลต่ำ', 'a': 'ผลไม้น้ำตาลต่ำ: สตรอว์เบอร์รี่, บลูเบอร์รี่, ฝรั่ง, แอปเปิ้ลเขียว, แคนตาลูป, แตงโม. หลีกเลี่ยงองุ่น, มะม่วงสุก, ลำไย.'},
    {'q': 'อาหารคลีนคืออะไร', 'a': 'อาหารคลีนคืออาหารที่ผ่านการแปรรูปน้อย ไม่ใส่ผงชูรส/น้ำตาลเติม ไม่ทอดในน้ำมันเยอะ เน้นผัก ผลไม้ ธัญพืชไม่ขัดสี และโปรตีนคุณภาพดี.'},
    {'q': 'Intermittent Fasting 16/8 คืออะไร', 'a': 'IF 16/8 = อดอาหาร 16 ชั่วโมง กินในช่วง 8 ชั่วโมง (เช่น 12:00-20:00). ช่วยควบคุมแคลอรี่และอินซูลิน เหมาะกับคนที่ไม่หิวเช้า.'},
    {'q': 'คาร์ดิโอหรือเวทดีกว่ากันสำหรับลดไขมัน', 'a': 'ทั้งสองจำเป็น: คาร์ดิโอเผาผลาญเยอะต่อครั้ง, เวทเพิ่มมวลกล้ามเนื้อทำให้ BMR สูงขึ้น. แนะนำคาร์ดิโอ 3 วัน + เวท 2 วัน/สัปดาห์.'},
    {'q': 'กินไก่ทอดแล้วน้ำหนักขึ้นเร็วไหม', 'a': 'ไก่ทอด 1 ชิ้น (120 กรัม) ~300-380 กิโลแคลอรี่ มีไขมันอิ่มตัวสูง กินบ่อยเกินจะเพิ่มน้ำหนักและ LDL แนะนำไม่เกิน 1 ครั้ง/สัปดาห์.'},
    {'q': 'หมาลาสูง sodium ไหม', 'a': 'หมาล่าเผ็ดหม่า 1 จาน (500 กรัม) มี sodium ประมาณ 2,500-3,500 มิลลิกรัม เกินขีดจำกัดวันละ 2,000 มก. ของ WHO แนะนำกินนาน ๆ ครั้ง.'},
    {'q': 'อาหารแคลต่ำอิ่มท้องมีอะไรบ้าง', 'a': 'ผักใบเขียวต้มจืด, ซุปใส, ไข่ขาว, อกไก่ต้ม, แตงกวา, บรอกโคลี, แอปเปิ้ล — กินแล้วอิ่มเพราะน้ำและไฟเบอร์สูง แต่แคลต่ำ.'},
    {'q': 'ถ้าเพิ่งกินชานมไข่มุกไป ต้องออกกำลังกายกี่นาที', 'a': 'ชานมไข่มุก 1 แก้ว 400 กิโลแคลอรี่ ต้องวิ่งระดับกลาง ~50 นาที หรือเดินเร็ว 100+ นาทีจึงเผาผลาญหมด.'},
    {'q': 'แพ้กลูเตนกินข้าวเจ้าได้ไหม', 'a': 'ข้าวเจ้าและข้าวเหนียวไม่มีกลูเตน กินได้. ควรเลี่ยง: ข้าวสาลี, บาร์เลย์, โอ๊ต (ยกเว้นติดฉลาก gluten-free), บะหมี่, ขนมปัง.'},
    {'q': 'แอปนี้คำนวณ BMR ยังไง', 'a': 'แอป Calories Guard ใช้สูตร Mifflin-St Jeor โดยต้องการข้อมูล เพศ อายุ ส่วนสูง น้ำหนัก ซึ่งกรอกตอนสมัครสมาชิกหรือแก้ไขที่หน้าโปรไฟล์.'},
    {'q': 'แอปเชื่อมกับ Samsung Health ได้ไหม', 'a': 'ได้ ผ่าน Health Connect บน Android — ไปที่หน้าบันทึกแคลอรี่ กดปุ่มซิงค์ Samsung Health แอปจะดึงแคลอรี่ที่เผาผลาญและจำนวนก้าวของวันนั้นมาอัตโนมัติ.'},
    {'q': 'ลบข้อมูลบัญชีในแอปทำยังไง', 'a': 'ไปที่หน้าโปรไฟล์ → ตั้งค่า → ลบบัญชี ข้อมูลทั้งหมดจะถูก soft-delete ทันทีและถูกลบถาวรใน 30 วัน ตามนโยบาย PDPA.'},
    {'q': 'AI ในแอปบอกแคลผิด ทำยังไง', 'a': 'ถ้าประมาณแคลไม่ตรง ให้กดแก้ไขที่รายการอาหารแล้วพิมพ์แคลจริง ระบบจะใช้ค่าที่แก้ไข และทีมแอดมินจะทบทวนเมนูให้แม่นขึ้นในรอบถัดไป.'},
    {'q': 'ทำไมไม่เห็นข้อมูลจาก Samsung Health', 'a': 'ตรวจสอบ: (1) ติดตั้ง Health Connect แล้ว (2) เปิดสิทธิ์ Samsung Health → Health Connect → อนุญาตทั้งหมด (3) มีกิจกรรมที่ Samsung Health บันทึกไว้จริงในวันนี้.'},
    {'q': 'ทำไมแอปไม่นับเนื้อสัตว์ดิบ', 'a': 'เนื้อดิบกับเนื้อปรุงสุกมีแคลต่างกันเพราะน้ำในเนื้อระเหย ระบบเก็บค่าเฉพาะเมนูที่ปรุงสุกเพื่อความแม่นยำในการคำนวณจริง.'},
    {'q': 'แอปใช้ AI อะไร', 'a': 'Calories Guard ใช้ LLM สำหรับประมาณแคลอรี่จากข้อความ ค้นหาเมนู และสร้างสูตรอาหารอัตโนมัติ รองรับ Gemini, DeepSeek และโมเดล open-source ที่ fine-tune เอง ผ่าน abstraction ในไฟล์ llm_provider.py.'},
    {'q': 'อาหารในตู้เย็นมีแค่ไข่กับผัก จะทำอะไรแคลต่ำได้', 'a': 'ไข่เจียวผัก 2 ฟอง (~250 kcal) หรือไข่ต้มคู่สลัดผัก (~160 kcal) หรือผักลวกจิ้มน้ำจิ้ม + ไข่ต้ม (~180 kcal) เป็นตัวเลือกแคลต่ำโปรตีนสูง.'},
    {'q': 'กินข้าวเย็นตอน 21:00 น้ำหนักขึ้นไหม', 'a': 'ไม่ใช่เวลาที่เป็นปัญหาหลัก แต่คือแคลรวมทั้งวัน. ถ้าแคลไม่เกิน TDEE และกินก่อนนอนอย่างน้อย 2 ชั่วโมง (เพื่อไม่กรดไหลย้อน) ก็ไม่ทำให้อ้วนโดยตรง.'},
]

print(f'Dataset: {len(QA_PAIRS)} Q&A pairs')

random.shuffle(QA_PAIRS)
split = int(0.85 * len(QA_PAIRS))
TRAIN_QA, VAL_QA = QA_PAIRS[:split], QA_PAIRS[split:]
print(f'Train: {len(TRAIN_QA)}, Val: {len(VAL_QA)}')

## 2. Thai word segmentation with an MLP

Thai has no spaces between words. We train a small MLP to predict whether each character position is the *start* of a new word, using a window of surrounding characters as features. pythainlp is used as the oracle tokenizer to generate labels.

This is a preprocessing sanity step — the DeepSeek tokenizer handles its own subword splitting, but seeing a learned segmenter work on our domain text is a useful sanity check on the dataset.

In [ ]:
from pythainlp.tokenize import word_tokenize

# Build char vocabulary from the dataset.
corpus = ''.join(f"{x['q']} {x['a']}" for x in QA_PAIRS)
chars = sorted(set(corpus))
char2id = {c: i + 1 for i, c in enumerate(chars)}  # 0 = padding
char2id['<PAD>'] = 0
V = len(char2id)
print('Char vocab size:', V)

In [ ]:
WINDOW = 7  # 3 chars before + current + 3 chars after

def make_char_dataset(texts):
    X, y = [], []
    for t in texts:
        tokens = word_tokenize(t, engine='newmm')
        # Reconstruct char-level boundary labels: 1 at the first char of each word, else 0.
        boundaries = set()
        cursor = 0
        for tok in tokens:
            boundaries.add(cursor)
            cursor += len(tok)
        for i, ch in enumerate(t):
            window_ids = []
            for off in range(-(WINDOW // 2), WINDOW // 2 + 1):
                j = i + off
                window_ids.append(char2id.get(t[j], 0) if 0 <= j < len(t) else 0)
            X.append(window_ids)
            y.append(1 if i in boundaries else 0)
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.float32)

all_texts = [x['q'] for x in QA_PAIRS] + [x['a'] for x in QA_PAIRS]
X_seg, y_seg = make_char_dataset(all_texts)
print('Segmentation samples:', len(X_seg), '| positive ratio:', float(y_seg.mean()))

In [ ]:
class WordBoundaryMLP(nn.Module):
    def __init__(self, vocab_size, emb_dim=24, hidden=64, window=WINDOW):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * window, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        e = self.emb(x)             # (B, W, D)
        flat = e.flatten(1)         # (B, W*D)
        return self.mlp(flat).squeeze(-1)

seg_model = WordBoundaryMLP(V + 1).to(DEVICE)
opt = torch.optim.Adam(seg_model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

EPOCHS_SEG = 10
BATCH = 256

perm_idx = torch.randperm(len(X_seg))
split_i = int(0.9 * len(X_seg))
Xtr, Xte = X_seg[perm_idx[:split_i]].to(DEVICE), X_seg[perm_idx[split_i:]].to(DEVICE)
ytr, yte = y_seg[perm_idx[:split_i]].to(DEVICE), y_seg[perm_idx[split_i:]].to(DEVICE)

seg_losses, seg_accs = [], []
for ep in range(EPOCHS_SEG):
    seg_model.train()
    epoch_loss = 0
    for i in range(0, len(Xtr), BATCH):
        xb, yb = Xtr[i:i+BATCH], ytr[i:i+BATCH]
        opt.zero_grad()
        logits = seg_model(xb)
        l = loss_fn(logits, yb)
        l.backward()
        opt.step()
        epoch_loss += l.item() * len(xb)
    seg_model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(seg_model(Xte)) > 0.5).float()
        acc = (preds == yte).float().mean().item()
    seg_losses.append(epoch_loss / len(Xtr))
    seg_accs.append(acc)
    print(f'Epoch {ep+1}/{EPOCHS_SEG}  loss={seg_losses[-1]:.4f}  val_acc={acc:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(seg_losses, marker='o'); ax[0].set_title('MLP word-boundary loss'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('BCE')
ax[1].plot(seg_accs, marker='o', color='green'); ax[1].set_title('Val accuracy'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('acc')
plt.tight_layout(); plt.show()

In [ ]:
def mlp_tokenize(text: str) -> list[str]:
    seg_model.eval()
    ids = []
    for i in range(len(text)):
        w = []
        for off in range(-(WINDOW // 2), WINDOW // 2 + 1):
            j = i + off
            w.append(char2id.get(text[j], 0) if 0 <= j < len(text) else 0)
        ids.append(w)
    with torch.no_grad():
        logits = seg_model(torch.tensor(ids, dtype=torch.long).to(DEVICE))
        boundaries = (torch.sigmoid(logits) > 0.5).cpu().tolist()
    tokens, cur = [], ''
    for ch, b in zip(text, boundaries):
        if b and cur:
            tokens.append(cur); cur = ''
        cur += ch
    if cur: tokens.append(cur)
    return tokens

for sample in ['ต้มยำกุ้งน้ำใสกี่แคลอรี่', 'แอปเชื่อมกับ Samsung Health ได้ไหม']:
    print(sample, '→', mlp_tokenize(sample))

## 3. Fine-tune DeepSeek with LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset

BASE_MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config if DEVICE == 'cuda' else None,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = 'คุณเป็นผู้ช่วยด้านโภชนาการและสุขภาพในแอป Calories Guard ตอบกระชับ เป็นภาษาไทย ให้ตัวเลขที่เชื่อถือได้เมื่อเป็นไปได้.'

def format_chat(q, a=None):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': q},
    ]
    if a is not None:
        msgs.append({'role': 'assistant', 'content': a})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=(a is None))

MAX_LEN = 512

def to_features(pair):
    text = format_chat(pair['q'], pair['a']) + tokenizer.eos_token
    out = tokenizer(text, truncation=True, max_length=MAX_LEN, padding='max_length')
    out['labels'] = out['input_ids'].copy()
    return out

train_ds = Dataset.from_list([to_features(x) for x in TRAIN_QA])
val_ds = Dataset.from_list([to_features(x) for x in VAL_QA])
print(train_ds)

## 4. Training loop with Q↔A alignment tracking

Every epoch we run the partially-trained model on the held-out validation split and compute two alignment metrics against the reference answer:

- **Token overlap %** — Jaccard of unique tokens between generated and reference answer.
- **Bigram overlap %** — same, on character bigrams (more robust for Thai since no word spaces).

Both are shown alongside training loss. A rising alignment curve against flat loss means we're fitting the answer shape, not memorizing.

In [ ]:
def jaccard(a: str, b: str, n=1) -> float:
    def grams(s):
        if n == 1:
            return set(mlp_tokenize(s))
        return {s[i:i+n] for i in range(len(s) - n + 1)}
    A, B = grams(a), grams(b)
    if not A and not B:
        return 1.0
    return len(A & B) / max(1, len(A | B))

@torch.no_grad()
def generate_answer(prompt, max_new=180):
    model.eval()
    text = format_chat(prompt)
    ids = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=False, temperature=0.0, pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    return gen.strip()

def eval_alignment():
    unigram, bigram, em = [], [], []
    for x in VAL_QA:
        pred = generate_answer(x['q'])
        unigram.append(jaccard(pred, x['a'], 1))
        bigram.append(jaccard(pred, x['a'], 2))
        em.append(1.0 if pred.strip() == x['a'].strip() else 0.0)
    return np.mean(unigram), np.mean(bigram), np.mean(em)

# Baseline before fine-tuning.
print('Baseline (epoch 0) alignment:', eval_alignment())

In [ ]:
EPOCHS = 5

train_losses, val_unigram, val_bigram, val_em = [], [], [], []

# Record the pre-training baseline so the plot shows "straight line" improvement from epoch 0.
u0, b0, e0 = eval_alignment()
val_unigram.append(u0); val_bigram.append(b0); val_em.append(e0)

for epoch in range(EPOCHS):
    args = TrainingArguments(
        output_dir='./out',
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy='no',
        bf16=(DEVICE == 'cuda'),
        report_to=[],
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    result = trainer.train()
    train_losses.append(result.training_loss)
    u, b, e = eval_alignment()
    val_unigram.append(u); val_bigram.append(b); val_em.append(e)
    print(f'Epoch {epoch+1}: loss={result.training_loss:.4f}  unigram%={u*100:.1f}  bigram%={b*100:.1f}  exact%={e*100:.1f}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4.5))
epochs_x = list(range(1, EPOCHS + 1))
epochs_eval = list(range(0, EPOCHS + 1))

ax1.plot(epochs_x, train_losses, '-o', color='#d76a3c', label='train loss')
ax1.set_xlabel('epoch')
ax1.set_ylabel('train loss', color='#d76a3c')
ax1.tick_params(axis='y', labelcolor='#d76a3c')

ax2 = ax1.twinx()
ax2.plot(epochs_eval, [u*100 for u in val_unigram], '-s', color='#628141', label='unigram overlap %')
ax2.plot(epochs_eval, [b*100 for b in val_bigram],  '-^', color='#2e7d32', label='bigram overlap %')
ax2.plot(epochs_eval, [e*100 for e in val_em],      '-d', color='#f9a825', label='exact match %')
ax2.set_ylabel('Q↔A alignment %')
ax2.set_ylim(0, 100)
ax2.grid(alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
plt.title('DeepSeek LoRA fine-tune — loss vs Q↔A alignment')
plt.tight_layout(); plt.show()

## 5. Save adapter

The adapter weights (~15-30 MB for this rank) are what you copy to the backend host. Set `LLM_PROVIDER=local`, `LOCAL_MODEL_PATH=deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`, and `LOCAL_ADAPTER_PATH=/path/to/calories_guard_adapter` — `ai_models/llm_provider.py` handles loading.

In [ ]:
ADAPTER_DIR = './calories_guard_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

with open(Path(ADAPTER_DIR) / 'calories_guard_meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': BASE_MODEL,
        'dataset_size': len(QA_PAIRS),
        'epochs': EPOCHS,
        'final_train_loss': float(train_losses[-1]),
        'final_unigram_alignment': float(val_unigram[-1]),
        'final_bigram_alignment': float(val_bigram[-1]),
        'final_exact_match': float(val_em[-1]),
    }, f, ensure_ascii=False, indent=2)

print('Saved adapter to', ADAPTER_DIR)

In [ ]:
# Sanity check — ask a held-out question and print the answer.
print('Q:', VAL_QA[0]['q'])
print('Expected:', VAL_QA[0]['a'])
print('Model   :', generate_answer(VAL_QA[0]['q']))